In [1]:
%%writefile app.py
# ===== 설정 =====
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Titanic 대시보드", layout="wide")

# ===== 함수 =====
def filter_data(df, sexes, classes, age_range, ports):
    f = df[df["sex"].isin(sexes)
           & df["pclass"].isin(classes)
           & df["embarked"].isin(ports)]
    if age_range != (0, 80):
        f = f[f["age"].between(*age_range)]
    return f

def kpi(df):
    if len(df) == 0:
        return 0, 0, 0
    n = len(df)
    rate = round(df["survived"].mean() * 100, 1)
    age = round(df["age"].mean(), 1)
    return n, rate, age

def class_rate_bar(df):
    r = (df.groupby("pclass")["survived"]
           .mean().mul(100).round(1)
           .reset_index())
    fig = px.bar(r, x="pclass", y="survived", text="survived",
                 title="객실 등급별 생존율(%)")
    return fig

def age_hist(df):
    fig = px.histogram(df, x="age", color="survived", barmode="overlay",
                       opacity=0.6, title="생존 여부별 나이 분포")
    return fig

LABELS = {"sex": "성별", "class": "객실 등급", "embarked": "승선 항구"}  # [추가] 한글 이름표

def group_rate_bar(df, col):                                   # [추가] ① 기준별 생존율 함수
    r = (df.groupby(col)["survived"]                           # [추가] 고른 기준(col)으로 묶기
           .mean().mul(100).round(1)                           # [추가] 생존율(%) 계산
           .reset_index())                                     # [추가]
    fig = px.bar(r, x=col, y="survived", text="survived",      # [추가] 막대 + 값 표시
                 title=f"{LABELS[col]}별 생존율(%)")            # [추가]
    return fig                                                 # [추가]

# ===== 데이터 =====
@st.cache_data
def load_data():
    return pd.read_csv("titanic.csv")

df = load_data()

# ===== 제목 =====
st.title("타이타닉, 누가 살아남았나?")

# ===== 사이드바 =====
sexes = st.sidebar.multiselect("성별", ["male", "female"], ["male", "female"])
classes = st.sidebar.multiselect("객실 등급", [1, 2, 3], [1, 2, 3])
ages = st.sidebar.slider("나이", 0, 80, (0, 80))
ports = st.sidebar.multiselect("승선 항구", ["S", "C", "Q"], ["S", "C", "Q"])

if not sexes or not classes or not ports:
    st.warning("성별, 등급, 항구를 하나 이상 선택하세요.")
    st.stop()

# ===== 필터 =====
f = filter_data(df, sexes, classes, ages, ports)
if len(f) == 0:
    st.warning("조건에 맞는 승객이 없습니다.")
    st.stop()

# ===== KPI =====
n, rate, age = kpi(f)
c1, c2, c3 = st.columns(3)
c1.metric("승객 수", f"{n}명")
c2.metric("생존율", f"{rate}%", f"{rate - 38.4:+.1f}%p")
c3.metric("평균 나이", f"{age}세")

# ===== 차트 · 표 =====
left, right = st.columns(2)
left.plotly_chart(class_rate_bar(f))
right.plotly_chart(age_hist(f))

# ===== 기준 선택 차트 =====                                   # [추가] ③ 기존 차트 아래
col = st.selectbox("비교 기준",                                 # [추가] ② selectbox
                   ["sex", "class", "embarked"],               # [추가]
                   format_func=lambda c: LABELS[c])            # [추가] 화면에는 한글로 표시
st.plotly_chart(group_rate_bar(f, col))                        # [추가]

with st.expander("데이터 보기"):
    st.dataframe(f)

Writing app.py


In [2]:
import seaborn as sns
sns.load_dataset("titanic").to_csv("titanic.csv", index=False)

In [3]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py &>/content/logs.txt & ./cloudflared tunnel --url http://localhost:8501

2026-11-24T01:38:29Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-24T01:38:29Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-24T01:38:32Z INF +--------------------------------------------------------------------------------------------+
2026-11-24T01:38:32Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-24T01:38:32Z INF |  https://recovery-recommends-lists-units.trycloudflare